In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 10: MLP Model for Time-Series Forecasting\n",
    "\n",
    "**Student Name:** M.Sufyan Khan  \n",
    "**Registration Number:** 22JZELE0477  \n",
    "**Course:** Machine Learning Lab  \n",
    "**Supervisor:** Engr. Irshad Ullah  \n",
    "**University:** UET Nowshera Campus  \n",
    "\n",
    "\n",
    "## Learning Objectives\n",
    "- Set the working directory and import required machine learning, TensorFlow, and utility modules.\n",
    "- Define an MLP neural network architecture for time-series input data.\n",
    "- Configure optimizer, checkpoints, and training monitor callbacks.\n",
    "- Load train, validation, and test CSV files for forecasting.\n",
    "- Train and evaluate the model using MAE, MSE, RMSE, R2, and explained variance score.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 1: Working Directory and Library Imports\n",
    "This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.chdir(r'D:\\Machine Learning\\ML Lab\\Lab 10')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score\n",
    "from timeseires.utils.to_split import to_split\n",
    "from timeseires.utils.multivariate_multi_step import multivariate_multi_step\n",
    "from timeseires.utils.multivariate_single_step import multivariate_single_step\n",
    "from timeseires.utils.univariate_multi_step import univariate_multi_step\n",
    "from timeseires.utils.univariate_single_step import univariate_single_step\n",
    "from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS\n",
    "from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint\n",
    "from tensorflow.keras.callbacks import ModelCheckpoint\n",
    "from timeseires.callbacks.TrainingMonitor import TrainingMonitor\n",
    "from tensorflow.keras.optimizers import Adam\n",
    "from tensorflow.keras.optimizers import SGD\n",
    "from tensorflow.keras.models import load_model\n",
    "from tensorflow.keras.layers import LSTM, Bidirectional, Add\n",
    "from tensorflow.keras.layers import BatchNormalization\n",
    "from tensorflow.keras.layers import Conv1D,TimeDistributed\n",
    "from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input\n",
    "from tensorflow.keras.models import Sequential,Model\n",
    "import pandas as pd\n",
    "import time, pickle\n",
    "import numpy as np\n",
    "import tensorflow.keras.backend as K\n",
    "import tensorflow\n",
    "from tensorflow.keras.layers import Input, Reshape, Lambda\n",
    "from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense\n",
    "from tensorflow.keras.regularizers import l2\n",
    "import glob\n",
    "import h5py\n",
    "import matplotlib.pyplot as plt\n",
    "from keras.callbacks import Callback"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "metadata": {},
   "outputs": [],
   "source": [
    "#lookback = 24\n",
    "model = None\n",
    "start_epoch = 0\n",
    "time_steps=24\n",
    "num_features=21"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "metadata": {},
   "outputs": [],
   "source": [
    "def MLP():\n",
    "    model = Sequential()\n",
    "    model.add(Flatten(input_shape=(time_steps , num_features)))\n",
    "    model.add(Dense(32, activation='relu'))\n",
    "    model.add(Dense(1))\n",
    "    return model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "metadata": {
    "scrolled": true
   },
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "c:\\Users\\Yousaf Tahir\\anaconda3\\Lib\\site-packages\\keras\\src\\layers\\reshaping\\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.\n",
      "  super().__init__(**kwargs)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\">Model: \"sequential\"</span>\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1mModel: \"sequential\"\u001b[0m\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\">┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃<span style=\"font-weight: bold\"> Layer (type)                    </span>┃<span style=\"font-weight: bold\"> Output Shape           </span>┃<span style=\"font-weight: bold\">       Param # </span>┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ flatten (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Flatten</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">504</span>)            │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                   │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">32</span>)             │        <span style=\"color: #00af00; text-decoration-color: #00af00\">16,160</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                 │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">1</span>)              │            <span style=\"color: #00af00; text-decoration-color: #00af00\">33</span> │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n",
       "</pre>\n"
      ],
      "text/plain": [
       "┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃\u001b[1m \u001b[0m\u001b[1mLayer (type)                   \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1mOutput Shape          \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1m      Param #\u001b[0m\u001b[1m \u001b[0m┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ flatten (\u001b[38;5;33mFlatten\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m504\u001b[0m)            │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (\u001b[38;5;33mDense\u001b[0m)                   │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m32\u001b[0m)             │        \u001b[38;5;34m16,160\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense_1 (\u001b[38;5;33mDense\u001b[0m)                 │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m1\u001b[0m)              │            \u001b[38;5;34m33\u001b[0m │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Total params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">16,193</span> (63.25 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Total params: \u001b[0m\u001b[38;5;34m16,193\u001b[0m (63.25 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">16,193</span> (63.25 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Trainable params: \u001b[0m\u001b[38;5;34m16,193\u001b[0m (63.25 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Non-trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> (0.00 B)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Non-trainable params: \u001b[0m\u001b[38;5;34m0\u001b[0m (0.00 B)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "model1 = MLP()\n",
    "model1.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 2: Model Parameters and MLP Architecture\n",
    "The following cells define time steps, number of features, and the MLP model structure used for forecasting.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "You must install pydot (`pip install pydot`) for `plot_model` to work.\n"
     ]
    }
   ],
   "source": [
    "tensorflow.keras.utils.plot_model(model1 )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'\n",
    "OUTPUT_PATH = r'D:\\Machine Learning\\ML Lab\\Lab 10'\n",
    "FIG_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.png\"])\n",
    "JSON_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.json\"])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "False"
      ]
     },
     "execution_count": 8,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "os.path.exists(JSON_PATH)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "metadata": {},
   "outputs": [],
   "source": [
    "# construct the callback to save only the *best* model to disk\n",
    "# based on the validation loss\n",
    "EpochCheckpoint1 = ModelCheckpoint(checkpoints,\n",
    "                             monitor=\"val_loss\",\n",
    "                             save_best_only=True, \n",
    "                             verbose=1)\n",
    "TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)\n",
    "\n",
    "# construct the set of callbacks\n",
    "callbacks = [EpochCheckpoint1,TrainingMonitor1]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] compiling model...\n",
      "WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.\n"
     ]
    }
   ],
   "source": [
    "# if there is no specific model checkpoint supplied, then initialize\n",
    "# the network and compile the model\n",
    "if model is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model =MLP()\n",
    "    opt = Adam(1e-3)\n",
    "    model.compile(loss= 'mae', optimizer=opt, metrics=[\"mae\", \"mape\"])\n",
    "# otherwise, load the checkpoint from disk\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model))\n",
    "    model = load_model(model)\n",
    "\n",
    "    # update the learning rate\n",
    "    print(\"[INFO] old learning rate: {}\".format(K.get_value(model.optimizer.lr)))\n",
    "    K.set_value(model.optimizer.lr, 1e-4)\n",
    "    print(\"[INFO] new learning rate: {}\".format(K.get_value(model.optimizer.lr)))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 3: Checkpoints, Callbacks, and Training Setup\n",
    "This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "metadata": {},
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "c:\\Users\\Yousaf Tahir\\anaconda3\\Lib\\site-packages\\sklearn\\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:\n",
      "https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations\n",
      "  warnings.warn(\n"
     ]
    },
    {
     "data": {
      "text/plain": [
       "((860, 21), (90, 21), (30, 21))"
      ]
     },
     "execution_count": 12,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "import os\n",
    "path_dataset =r'D:\\Machine Learning\\ML Lab\\Lab 10'\n",
    "path_tr = os.path.join(path_dataset, 'train.csv')\n",
    "df_tr = pd.read_csv(path_tr)\n",
    "train_set = df_tr.iloc[:].values\n",
    "path_v = os.path.join(path_dataset, 'validation.csv')\n",
    "df_v = pd.read_csv(path_v)\n",
    "validation_set = df_v.iloc[:].values \n",
    "path_te = os.path.join(path_dataset, 'test.csv')\n",
    "df_te = pd.read_csv(path_te)\n",
    "test_set = df_te.iloc[:].values \n",
    "\n",
    "path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')\n",
    "scaler = pickle.load(open(path_scaler, 'rb'))\n",
    "\n",
    "train_set.shape, validation_set.shape, test_set.shape"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Time Consumed 0.0034329891204833984 sec\n"
     ]
    }
   ],
   "source": [
    "start = time.time()\n",
    "train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)\n",
    "validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)\n",
    "test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)\n",
    "print('Time Consumed', time.time()-start, \"sec\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "(835, 24, 21)"
      ]
     },
     "execution_count": 14,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "train_X.shape"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m17s\u001b[0m 660ms/step - loss: 0.3598 - mae: 0.3598 - mape: 137.9386\n",
      "Epoch 1: val_loss improved from None to 0.06231, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0001-loss0.06.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0001-loss0.06.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 8ms/step - loss: 0.1632 - mae: 0.1632 - mape: 80.1163 - val_loss: 0.0623 - val_mae: 0.0623 - val_mape: 18.7063\n",
      "Epoch 2/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 17ms/step - loss: 0.0999 - mae: 0.0999 - mape: 32.0019\n",
      "Epoch 2: val_loss improved from 0.06231 to 0.05801, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0002-loss0.06.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 2: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0002-loss0.06.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0837 - mae: 0.0837 - mape: 42.5397 - val_loss: 0.0580 - val_mae: 0.0580 - val_mape: 20.2650\n",
      "Epoch 3/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0734 - mae: 0.0734 - mape: 46.9744\n",
      "Epoch 3: val_loss did not improve from 0.05801\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0690 - mae: 0.0690 - mape: 39.9181 - val_loss: 0.0834 - val_mae: 0.0834 - val_mape: 27.5543\n",
      "Epoch 4/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0431 - mae: 0.0431 - mape: 28.7699\n",
      "Epoch 4: val_loss did not improve from 0.05801\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0690 - mae: 0.0690 - mape: 38.0091 - val_loss: 0.0656 - val_mae: 0.0656 - val_mape: 22.8119\n",
      "Epoch 5/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.1011 - mae: 0.1011 - mape: 254.9201\n",
      "Epoch 5: val_loss did not improve from 0.05801\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 16ms/step - loss: 0.0690 - mae: 0.0690 - mape: 38.9874 - val_loss: 0.0805 - val_mae: 0.0805 - val_mape: 25.9182\n",
      "Epoch 6/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0815 - mae: 0.0815 - mape: 24.6569\n",
      "Epoch 6: val_loss did not improve from 0.05801\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0739 - mae: 0.0739 - mape: 41.0248 - val_loss: 0.0582 - val_mae: 0.0582 - val_mape: 18.1730\n",
      "Epoch 7/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 26ms/step - loss: 0.0493 - mae: 0.0493 - mape: 16.8445\n",
      "Epoch 7: val_loss did not improve from 0.05801\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0542 - mae: 0.0542 - mape: 26.5686 - val_loss: 0.0750 - val_mae: 0.0750 - val_mape: 23.7514\n",
      "Epoch 8/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0559 - mae: 0.0559 - mape: 24.8313\n",
      "Epoch 8: val_loss improved from 0.05801 to 0.05367, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0008-loss0.05.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 8: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0008-loss0.05.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0563 - mae: 0.0563 - mape: 27.7922 - val_loss: 0.0537 - val_mae: 0.0537 - val_mape: 16.5497\n",
      "Epoch 9/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0608 - mae: 0.0608 - mape: 21.4274\n",
      "Epoch 9: val_loss did not improve from 0.05367\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0568 - mae: 0.0568 - mape: 28.6445 - val_loss: 0.0564 - val_mae: 0.0564 - val_mape: 18.4923\n",
      "Epoch 10/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0354 - mae: 0.0354 - mape: 10.1235\n",
      "Epoch 10: val_loss did not improve from 0.05367\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0462 - mae: 0.0462 - mape: 22.8103 - val_loss: 0.1120 - val_mae: 0.1120 - val_mape: 38.8564\n",
      "Epoch 11/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0912 - mae: 0.0912 - mape: 28.7823\n",
      "Epoch 11: val_loss did not improve from 0.05367\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0565 - mae: 0.0565 - mape: 28.9143 - val_loss: 0.0677 - val_mae: 0.0677 - val_mape: 23.6436\n",
      "Epoch 12/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0583 - mae: 0.0583 - mape: 18.5080\n",
      "Epoch 12: val_loss did not improve from 0.05367\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0487 - mae: 0.0487 - mape: 25.4021 - val_loss: 0.0706 - val_mae: 0.0706 - val_mape: 23.4447\n",
      "Epoch 13/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.1172 - mae: 0.1172 - mape: 54.0592\n",
      "Epoch 13: val_loss did not improve from 0.05367\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0832 - mae: 0.0832 - mape: 40.8068 - val_loss: 0.0737 - val_mae: 0.0737 - val_mape: 25.4909\n",
      "Epoch 14/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 25ms/step - loss: 0.0919 - mae: 0.0919 - mape: 30.7871\n",
      "Epoch 14: val_loss improved from 0.05367 to 0.04973, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0014-loss0.05.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 14: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0014-loss0.05.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0540 - mae: 0.0540 - mape: 28.9815 - val_loss: 0.0497 - val_mae: 0.0497 - val_mape: 16.2667\n",
      "Epoch 15/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0402 - mae: 0.0402 - mape: 12.0981\n",
      "Epoch 15: val_loss did not improve from 0.04973\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0405 - mae: 0.0405 - mape: 20.0557 - val_loss: 0.0563 - val_mae: 0.0563 - val_mape: 17.5303\n",
      "Epoch 16/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 28ms/step - loss: 0.0647 - mae: 0.0647 - mape: 30.3298\n",
      "Epoch 16: val_loss did not improve from 0.04973\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0602 - mae: 0.0602 - mape: 30.1953 - val_loss: 0.0533 - val_mae: 0.0533 - val_mape: 17.1092\n",
      "Epoch 17/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0371 - mae: 0.0371 - mape: 26.5946\n",
      "Epoch 17: val_loss improved from 0.04973 to 0.04048, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0017-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 17: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0017-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0498 - mae: 0.0498 - mape: 24.6434 - val_loss: 0.0405 - val_mae: 0.0405 - val_mape: 13.0794\n",
      "Epoch 18/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0334 - mae: 0.0334 - mape: 10.9010\n",
      "Epoch 18: val_loss did not improve from 0.04048\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0374 - mae: 0.0374 - mape: 18.3587 - val_loss: 0.0524 - val_mae: 0.0524 - val_mape: 18.2301\n",
      "Epoch 19/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 24ms/step - loss: 0.0326 - mae: 0.0326 - mape: 10.6840\n",
      "Epoch 19: val_loss improved from 0.04048 to 0.03854, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0019-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 19: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0019-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0385 - mae: 0.0385 - mape: 19.0874 - val_loss: 0.0385 - val_mae: 0.0385 - val_mape: 12.4646\n",
      "Epoch 20/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0309 - mae: 0.0309 - mape: 13.2601\n",
      "Epoch 20: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0335 - mae: 0.0335 - mape: 19.5699 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 11.5752\n",
      "Epoch 21/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0350 - mae: 0.0350 - mape: 13.1060\n",
      "Epoch 21: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0431 - mae: 0.0431 - mape: 21.0205 - val_loss: 0.0459 - val_mae: 0.0459 - val_mape: 16.3140\n",
      "Epoch 22/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0419 - mae: 0.0419 - mape: 14.4775\n",
      "Epoch 22: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0412 - mae: 0.0412 - mape: 22.8567 - val_loss: 0.0602 - val_mae: 0.0602 - val_mape: 16.6723\n",
      "Epoch 23/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0392 - mae: 0.0392 - mape: 25.1110\n",
      "Epoch 23: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0391 - mae: 0.0391 - mape: 17.5927 - val_loss: 0.0408 - val_mae: 0.0408 - val_mape: 12.3588\n",
      "Epoch 24/60\n",
      "\u001b[1m26/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━\u001b[0m \u001b[1m0s\u001b[0m 2ms/step - loss: 0.0315 - mae: 0.0315 - mape: 17.1804\n",
      "Epoch 24: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 9ms/step - loss: 0.0314 - mae: 0.0314 - mape: 17.1540 - val_loss: 0.0463 - val_mae: 0.0463 - val_mape: 14.4823\n",
      "Epoch 25/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 24ms/step - loss: 0.0613 - mae: 0.0613 - mape: 73.7090\n",
      "Epoch 25: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0401 - mae: 0.0401 - mape: 22.1629 - val_loss: 0.0389 - val_mae: 0.0389 - val_mape: 12.0616\n",
      "Epoch 26/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0351 - mae: 0.0351 - mape: 9.3575\n",
      "Epoch 26: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0357 - mae: 0.0357 - mape: 15.3648 - val_loss: 0.0522 - val_mae: 0.0522 - val_mape: 18.1320\n",
      "Epoch 27/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0554 - mae: 0.0554 - mape: 15.4071\n",
      "Epoch 27: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0416 - mae: 0.0416 - mape: 20.8998 - val_loss: 0.0671 - val_mae: 0.0671 - val_mape: 20.7104\n",
      "Epoch 28/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0774 - mae: 0.0774 - mape: 23.8699\n",
      "Epoch 28: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0491 - mae: 0.0491 - mape: 24.6353 - val_loss: 0.0458 - val_mae: 0.0458 - val_mape: 15.8110\n",
      "Epoch 29/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0349 - mae: 0.0349 - mape: 13.9113\n",
      "Epoch 29: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0332 - mae: 0.0332 - mape: 16.3798 - val_loss: 0.0680 - val_mae: 0.0680 - val_mape: 20.1629\n",
      "Epoch 30/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0714 - mae: 0.0714 - mape: 18.9954\n",
      "Epoch 30: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0431 - mae: 0.0431 - mape: 19.4215 - val_loss: 0.0509 - val_mae: 0.0509 - val_mape: 17.0641\n",
      "Epoch 31/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0428 - mae: 0.0428 - mape: 20.2084\n",
      "Epoch 31: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0401 - mae: 0.0401 - mape: 22.6761 - val_loss: 0.0485 - val_mae: 0.0485 - val_mape: 16.3239\n",
      "Epoch 32/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0366 - mae: 0.0366 - mape: 18.0982\n",
      "Epoch 32: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 9ms/step - loss: 0.0391 - mae: 0.0391 - mape: 19.2891 - val_loss: 0.0432 - val_mae: 0.0432 - val_mape: 14.0740\n",
      "Epoch 33/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 24ms/step - loss: 0.0446 - mae: 0.0446 - mape: 18.5806\n",
      "Epoch 33: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0393 - mae: 0.0393 - mape: 20.0890 - val_loss: 0.0450 - val_mae: 0.0450 - val_mape: 13.9641\n",
      "Epoch 34/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0304 - mae: 0.0304 - mape: 11.7541\n",
      "Epoch 34: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0380 - mae: 0.0380 - mape: 18.4835 - val_loss: 0.0398 - val_mae: 0.0398 - val_mape: 12.1026\n",
      "Epoch 35/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0277 - mae: 0.0277 - mape: 11.8278\n",
      "Epoch 35: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0278 - mae: 0.0278 - mape: 13.3920 - val_loss: 0.0433 - val_mae: 0.0433 - val_mape: 13.9626\n",
      "Epoch 36/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 25ms/step - loss: 0.0263 - mae: 0.0263 - mape: 12.1507\n",
      "Epoch 36: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0339 - mae: 0.0339 - mape: 18.9161 - val_loss: 0.1065 - val_mae: 0.1065 - val_mape: 34.1348\n",
      "Epoch 37/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0847 - mae: 0.0847 - mape: 29.7392\n",
      "Epoch 37: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0435 - mae: 0.0435 - mape: 22.0114 - val_loss: 0.0735 - val_mae: 0.0735 - val_mape: 21.8525\n",
      "Epoch 38/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0815 - mae: 0.0815 - mape: 62.1208\n",
      "Epoch 38: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0395 - mae: 0.0395 - mape: 19.4335 - val_loss: 0.0428 - val_mae: 0.0428 - val_mape: 11.8967\n",
      "Epoch 39/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0243 - mae: 0.0243 - mape: 10.3601\n",
      "Epoch 39: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0331 - mae: 0.0331 - mape: 15.8314 - val_loss: 0.0529 - val_mae: 0.0529 - val_mape: 14.6093\n",
      "Epoch 40/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0372 - mae: 0.0372 - mape: 11.5028\n",
      "Epoch 40: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0336 - mae: 0.0336 - mape: 15.9776 - val_loss: 0.0647 - val_mae: 0.0647 - val_mape: 20.9709\n",
      "Epoch 41/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0394 - mae: 0.0394 - mape: 32.9758\n",
      "Epoch 41: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0305 - mae: 0.0305 - mape: 16.4496 - val_loss: 0.0543 - val_mae: 0.0543 - val_mape: 16.3748\n",
      "Epoch 42/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0352 - mae: 0.0352 - mape: 16.3848\n",
      "Epoch 42: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0339 - mae: 0.0339 - mape: 16.7022 - val_loss: 0.0593 - val_mae: 0.0593 - val_mape: 19.0245\n",
      "Epoch 43/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0468 - mae: 0.0468 - mape: 22.8367\n",
      "Epoch 43: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0317 - mae: 0.0317 - mape: 14.3458 - val_loss: 0.0761 - val_mae: 0.0761 - val_mape: 22.9505\n",
      "Epoch 44/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0636 - mae: 0.0636 - mape: 28.6217\n",
      "Epoch 44: val_loss did not improve from 0.03854\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0432 - mae: 0.0432 - mape: 18.9361 - val_loss: 0.0709 - val_mae: 0.0709 - val_mape: 22.7552\n",
      "Epoch 45/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0395 - mae: 0.0395 - mape: 24.0715\n",
      "Epoch 45: val_loss improved from 0.03854 to 0.03387, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 45: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0349 - mae: 0.0349 - mape: 16.2079 - val_loss: 0.0339 - val_mae: 0.0339 - val_mape: 10.0437\n",
      "Epoch 46/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 17ms/step - loss: 0.0329 - mae: 0.0329 - mape: 12.7789\n",
      "Epoch 46: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0373 - mae: 0.0373 - mape: 19.8820 - val_loss: 0.0612 - val_mae: 0.0612 - val_mape: 20.1775\n",
      "Epoch 47/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0386 - mae: 0.0386 - mape: 12.8680\n",
      "Epoch 47: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0342 - mae: 0.0342 - mape: 20.5002 - val_loss: 0.0471 - val_mae: 0.0471 - val_mape: 12.6986\n",
      "Epoch 48/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0306 - mae: 0.0306 - mape: 16.8225\n",
      "Epoch 48: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0294 - mae: 0.0294 - mape: 16.2543 - val_loss: 0.0394 - val_mae: 0.0394 - val_mape: 11.4059\n",
      "Epoch 49/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0372 - mae: 0.0372 - mape: 10.0116\n",
      "Epoch 49: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0314 - mae: 0.0314 - mape: 15.0991 - val_loss: 0.0417 - val_mae: 0.0417 - val_mape: 11.5192\n",
      "Epoch 50/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0256 - mae: 0.0256 - mape: 38.8252\n",
      "Epoch 50: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0308 - mae: 0.0308 - mape: 17.8896 - val_loss: 0.0542 - val_mae: 0.0542 - val_mape: 17.5205\n",
      "Epoch 51/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0594 - mae: 0.0594 - mape: 29.9934\n",
      "Epoch 51: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0361 - mae: 0.0361 - mape: 19.0491 - val_loss: 0.0383 - val_mae: 0.0383 - val_mape: 10.6746\n",
      "Epoch 52/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0253 - mae: 0.0253 - mape: 10.3234\n",
      "Epoch 52: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0321 - mae: 0.0321 - mape: 16.7585 - val_loss: 0.0484 - val_mae: 0.0484 - val_mape: 13.5548\n",
      "Epoch 53/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0334 - mae: 0.0334 - mape: 36.4479\n",
      "Epoch 53: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0293 - mae: 0.0293 - mape: 14.7821 - val_loss: 0.0804 - val_mae: 0.0804 - val_mape: 23.6100\n",
      "Epoch 54/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0339 - mae: 0.0339 - mape: 17.4074\n",
      "Epoch 54: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0363 - mae: 0.0363 - mape: 15.8591 - val_loss: 0.0585 - val_mae: 0.0585 - val_mape: 18.2115\n",
      "Epoch 55/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0822 - mae: 0.0822 - mape: 23.8336\n",
      "Epoch 55: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0402 - mae: 0.0402 - mape: 19.5294 - val_loss: 0.0465 - val_mae: 0.0465 - val_mape: 13.2285\n",
      "Epoch 56/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0386 - mae: 0.0386 - mape: 14.6138\n",
      "Epoch 56: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0316 - mae: 0.0316 - mape: 15.5943 - val_loss: 0.0539 - val_mae: 0.0539 - val_mape: 18.1170\n",
      "Epoch 57/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0599 - mae: 0.0599 - mape: 20.9127\n",
      "Epoch 57: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0356 - mae: 0.0356 - mape: 18.1217 - val_loss: 0.0395 - val_mae: 0.0395 - val_mape: 10.9015\n",
      "Epoch 58/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0317 - mae: 0.0317 - mape: 13.9774\n",
      "Epoch 58: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0291 - mae: 0.0291 - mape: 13.9620 - val_loss: 0.0732 - val_mae: 0.0732 - val_mape: 23.3769\n",
      "Epoch 59/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0528 - mae: 0.0528 - mape: 17.0970\n",
      "Epoch 59: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0365 - mae: 0.0365 - mape: 19.4550 - val_loss: 0.0471 - val_mae: 0.0471 - val_mape: 16.7886\n",
      "Epoch 60/60\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0434 - mae: 0.0434 - mape: 33.0155\n",
      "Epoch 60: val_loss did not improve from 0.03387\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0385 - mae: 0.0385 - mape: 19.6228 - val_loss: 0.0570 - val_mae: 0.0570 - val_mape: 17.7414\n"
     ]
    }
   ],
   "source": [
    "epochs = 60\n",
    "verbose = 1 #0\n",
    "batch_size = 32\n",
    "History = model.fit(train_X,\n",
    "                        train_y,\n",
    "                        batch_size=batch_size,   \n",
    "                        epochs = epochs, \n",
    "                        validation_data = (validation_X,validation_y),\n",
    "                        callbacks=callbacks,\n",
    "                    verbose = verbose)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 51ms/step\n",
      "Mean Absolute Error (MAE): 740.81\n",
      "Median Absolute Error (MedAE): 795.21\n",
      "Mean Squared Error (MSE): 730659.8\n",
      "Root Mean Squared Error (RMSE): 854.79\n",
      "Mean Absolute Percentage Error (MAPE): 4.74 %\n",
      "Median Absolute Percentage Error (MDAPE): 5.06 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape=  (5, 1)\n",
      "y_pred.shape=  (5, 1)\n"
     ]
    }
   ],
   "source": [
    "model = load_model(r'D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5', compile=False)\n",
    "\n",
    "y_pred_scaled   = model.predict(test_X)\n",
    "y_pred          = scaler.inverse_transform(y_pred_scaled)\n",
    "y_test_unscaled = scaler.inverse_transform(test_y)\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(abs(y_pred - y_test_unscaled)) \n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squarred Error (RMSE) \n",
    "RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape= ',y_test_unscaled.shape)\n",
    "print('y_pred.shape= ',y_pred.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 4: Dataset Loading and Forecast Evaluation\n",
    "The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fine Tuning"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'\n",
    "model=r'D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5\\E1-cp-0055-loss0.03.h5'\n",
    "start_epoch= 56"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] loading D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5...\n",
      "[INFO] learning rate: 9.999999747378752e-05\n"
     ]
    }
   ],
   "source": [
    "# construct the callback to save only the best model to disk\n",
    "EpochCheckpoint1 = ModelCheckpoint(\n",
    "    checkpoints,\n",
    "    monitor=\"val_loss\",\n",
    "    save_best_only=True,\n",
    "    verbose=1\n",
    ")\n",
    "\n",
    "TrainingMonitor1 = TrainingMonitor(\n",
    "    FIG_PATH,\n",
    "    jsonPath=JSON_PATH,\n",
    "    startAt=start_epoch\n",
    ")\n",
    "\n",
    "callbacks = [EpochCheckpoint1, TrainingMonitor1]\n",
    "\n",
    "model_path = r'D:\\Machine Learning\\ML Lab\\Lab 10\\E1-cp-0045-loss0.03.h5'\n",
    "# model_path = None   # use this if you want to train from scratch\n",
    "\n",
    "if model_path is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model = PC.build(time_steps=24, num_features=21, reg=0.0005)\n",
    "\n",
    "    opt = Adam(learning_rate=1e-3)\n",
    "    model.compile(\n",
    "        loss=\"mae\",\n",
    "        optimizer=opt,\n",
    "        metrics=[\"mae\", \"mape\"]\n",
    "    )\n",
    "\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model_path))\n",
    "    model = load_model(model_path, compile=False)\n",
    "\n",
    "    opt = Adam(learning_rate=1e-4)\n",
    "    model.compile(\n",
    "        loss=\"mae\",\n",
    "        optimizer=opt,\n",
    "        metrics=[\"mae\", \"mape\"]\n",
    "    )\n",
    "\n",
    "    print(\"[INFO] learning rate: {}\".format(K.get_value(model.optimizer.learning_rate)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 25,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/10\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m13s\u001b[0m 526ms/step - loss: 0.0350 - mae: 0.0350 - mape: 32.9010\n",
      "Epoch 1: val_loss improved from None to 0.03790, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-0001-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-0001-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 11ms/step - loss: 0.0224 - mae: 0.0224 - mape: 10.5835 - val_loss: 0.0379 - val_mae: 0.0379 - val_mape: 10.1626\n",
      "Epoch 2/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0231 - mae: 0.0231 - mape: 6.2908\n",
      "Epoch 2: val_loss did not improve from 0.03790\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0208 - mae: 0.0208 - mape: 8.9070 - val_loss: 0.0446 - val_mae: 0.0446 - val_mape: 13.0990\n",
      "Epoch 3/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0196 - mae: 0.0196 - mape: 5.8012\n",
      "Epoch 3: val_loss did not improve from 0.03790\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0197 - mae: 0.0197 - mape: 8.6214 - val_loss: 0.0423 - val_mae: 0.0423 - val_mape: 12.0735\n",
      "Epoch 4/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0172 - mae: 0.0172 - mape: 7.0372\n",
      "Epoch 4: val_loss improved from 0.03790 to 0.03779, saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-0004-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 4: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-0004-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0199 - mae: 0.0199 - mape: 8.8785 - val_loss: 0.0378 - val_mae: 0.0378 - val_mape: 10.7654\n",
      "Epoch 5/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 22ms/step - loss: 0.0182 - mae: 0.0182 - mape: 5.7300\n",
      "Epoch 5: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0199 - mae: 0.0199 - mape: 8.4516 - val_loss: 0.0392 - val_mae: 0.0392 - val_mape: 11.3105\n",
      "Epoch 6/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0152 - mae: 0.0152 - mape: 6.3117\n",
      "Epoch 6: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0193 - mae: 0.0193 - mape: 8.0188 - val_loss: 0.0428 - val_mae: 0.0428 - val_mape: 12.8140\n",
      "Epoch 7/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 31ms/step - loss: 0.0205 - mae: 0.0205 - mape: 8.6106\n",
      "Epoch 7: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0187 - mae: 0.0187 - mape: 7.7560 - val_loss: 0.0432 - val_mae: 0.0432 - val_mape: 12.5146\n",
      "Epoch 8/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0170 - mae: 0.0170 - mape: 6.3349\n",
      "Epoch 8: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0188 - mae: 0.0188 - mape: 8.4571 - val_loss: 0.0418 - val_mae: 0.0418 - val_mape: 12.0557\n",
      "Epoch 9/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0180 - mae: 0.0180 - mape: 5.8705\n",
      "Epoch 9: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0188 - mae: 0.0188 - mape: 8.1544 - val_loss: 0.0404 - val_mae: 0.0404 - val_mape: 11.5008\n",
      "Epoch 10/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 23ms/step - loss: 0.0138 - mae: 0.0138 - mape: 4.8563\n",
      "Epoch 10: val_loss did not improve from 0.03779\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0194 - mae: 0.0194 - mape: 8.6199 - val_loss: 0.0403 - val_mae: 0.0403 - val_mape: 11.6690\n"
     ]
    }
   ],
   "source": [
    "epochs = 10\n",
    "verbose = 1 #0\n",
    "batch_size = 32\n",
    "History = model.fit(train_X,\n",
    "                        train_y,\n",
    "                        batch_size=batch_size,   \n",
    "                        epochs = epochs, \n",
    "                        validation_data = (validation_X,validation_y),\n",
    "                        callbacks=callbacks,\n",
    "                        verbose = verbose)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 26,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 40ms/step\n",
      "Mean Absolute Error (MAE): 688.51\n",
      "Median Absolute Error (MedAE): 535.13\n",
      "Mean Squared Error (MSE): 674808.56\n",
      "Root Mean Squared Error (RMSE): 821.47\n",
      "Mean Absolute Percentage Error (MAPE): 4.42 %\n",
      "Median Absolute Percentage Error (MDAPE): 3.4 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape=  (5, 1)\n",
      "y_pred.shape=  (5, 1)\n"
     ]
    }
   ],
   "source": [
    "model = load_model(r'D:\\Machine Learning\\ML Lab\\Lab 10\\\\E2-cp-0004-loss0.04.h5', compile=False)\n",
    "\n",
    "y_pred_scaled   = model.predict(test_X)\n",
    "y_pred          = scaler.inverse_transform(y_pred_scaled)\n",
    "y_test_unscaled = scaler.inverse_transform(test_y)\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(abs(y_pred - y_test_unscaled)) \n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squarred Error (RMSE) \n",
    "RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape= ',y_test_unscaled.shape)\n",
    "print('y_pred.shape= ',y_pred.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Final Conclusion\n",
    "In this lab, an MLP neural network was implemented for time-series forecasting. The notebook demonstrates model creation, callback configuration, training workflow, and regression-based evaluation."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "base",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}